# LLM 5일차 (3) — 하이브리드 검색 : BM25 + 임베딩 · RRF(상호 순위 융합) · `EnsembleRetriever`

수업 필기 `RAG_실습5_1`(기술 지원 매뉴얼로 하이브리드 RAG 챗봇)을 정리 · 개선한 노트입니다. (교재 `AI Agent` **p.62~64** — 하이브리드 검색 · 단일 검색의 문제점 · RRF · EnsembleRetriever)

```python
hybrid_retriever = EnsembleRetriever(retrievers=[sparse_retriever, dense_retriever], weights=[0.5, 0.5])
hybrid_retriever.invoke('ERR-502 오류코드 조치법')     # 두 검색기의 '순위' 를 RRF 로 합친다
```

> **출력 읽는 법** — 정리 노트북은 **API 키 없이** 만들었다(Day 046 ~ 049 와 같은 방식).
>
> | 표시 | 뜻 |
> |---|---|
> | `# 📼` | **수업 시간에 실제로 실행한 결과**를 필기 원본에서 그대로 옮긴 출력 |
> | `# ⏳` | API 키가 필요해 **출력이 비어 있는 셀** — 직접 실행 |
> | `# 🧪` | OpenAI 대신 **로컬 임베딩 모델**(`jhgan/ko-sroberta-multitask`) 또는 **가짜 LLM** 으로 실제로 돌린 셀 — 밀집 검색 순위 · 점수는 수업의 `text-embedding-3-small` 과 다를 수 있다 |
> | 표시 없음 | 정리 노트북에서 실제로 실행한 셀 (필터 · BM25 · 형태소 분석 · RRF 손계산 · 수업 기록 분석) — **BM25 는 임베딩이 필요 없어서 수업과 같은 결과** |

## 0. 설치 · 실행 방법

| 패키지 | 이 노트에서 쓰는 것 | 설치 |
|---|---|---|
| `langchain-classic` | `EnsembleRetriever` (LangChain 1.0 부터 이 패키지로 옮겨짐) | `pip install langchain-classic` |
| `langchain-community` · `rank_bm25` · `python-mecab-ko` | BM25 희소 검색 | `pip install langchain-community rank_bm25 python-mecab-ko` |
| `langchain-chroma` · `langchain-huggingface` · `sentence-transformers` | 밀집 검색 (🧪 로컬 임베딩) | `pip install langchain-chroma langchain-huggingface sentence-transformers` |
| `langchain-openai` · `python-dotenv` · `pandas` | ⏳ `OpenAIEmbeddings` · `ChatOpenAI` | `pip install langchain-openai python-dotenv pandas` |

**데이터** : 같은 폴더의 `tech_support_manual.txt`. **실행** : 맨 위부터 모두 실행(🧪 로컬 임베딩 모델 약 440MB 를 처음 한 번 받음).

In [1]:
# ① 설치 (처음 한 번)
# !pip install -q langchain-classic langchain-community rank_bm25 python-mecab-ko langchain-chroma langchain-huggingface sentence-transformers langchain-openai python-dotenv pandas

In [2]:
import os, shutil, warnings
import numpy as np
import pandas as pd
warnings.filterwarnings('ignore', message='.*sunset.*')
from langchain_core.documents import Document
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.retrievers import BM25Retriever
from langchain_chroma import Chroma
from langchain_classic.retrievers import EnsembleRetriever
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

for d in ['chroma_db_tech_support_local', 'chroma_db_error_codes_local']:
    shutil.rmtree(d, ignore_errors=True)

In [3]:
# .env 파일 → 환경 변수 (키 값은 절대 print 하지 않는다)
from dotenv import load_dotenv, find_dotenv
loaded = load_dotenv(find_dotenv(usecwd=True))
print('.env 읽음 :', loaded, '| OPENAI_API_KEY 있음 :', bool(os.getenv('OPENAI_API_KEY')))

.env 읽음 : False | OPENAI_API_KEY 있음 : False


## 1. 데이터 · 분할 — 가장 긴 QA 쌍 208자 → `chunk_size=210`

In [4]:
documents = TextLoader(file_path='tech_support_manual.txt', encoding='utf-8').load()
qa_pairs = documents[0].page_content.split('\n\n')
print('QA 쌍 글자 수 :', [len(p) for p in qa_pairs], '→ 최대', max(len(p) for p in qa_pairs))
for size in [208, 209, 210]:
    n = len(RecursiveCharacterTextSplitter(chunk_size=size, chunk_overlap=0).split_documents(documents))
    print(f'  chunk_size={size} → 청크 {n}개')
docs = RecursiveCharacterTextSplitter(chunk_size=210, chunk_overlap=0).split_documents(documents)
for i, d in enumerate(docs):
    d.metadata['qa'] = i + 1                    # 몇 번째 QA 인지 (결과를 읽기 쉽게)

QA 쌍 글자 수 : [204, 208, 184, 146, 145, 171, 174, 153] → 최대 208
  chunk_size=208 → 청크 9개
  chunk_size=209 → 청크 9개
  chunk_size=210 → 청크 8개


- 수업 값 210 은 **딱 맞는 최솟값**이다 — 구분자 `\n\n` 2글자가 붙어 208 + 2 = 210 (Day 048 의 `keep_separator`). 208 · 209 로 하면 가장 긴 쌍이 갈라진다. QA 를 하나 더 넣으면 다시 재야 하니 조금 여유를 두는 편이 안전하다

## 2. 희소 검색기 · 밀집 검색기

In [5]:
from mecab import MeCab
mecab = MeCab()

def mecab_tokenizer(text):
    '''수업 함수 : 품사가 N(명사) · V(용언) · S(기호 · 외국어 · 숫자) 로 시작하는 형태소만'''
    return [morph for morph, pos in mecab.pos(text) if pos.startswith(('N', 'V', 'S'))]

sparse_retriever = BM25Retriever.from_documents(documents=docs, preprocess_func=mecab_tokenizer, k=2)

In [ ]:
# 📼 아래 출력 = 수업 시간에 실행한 결과 (필기 원본에서 옮김)

test_query = "ERR-502 오류코드 조치법"
retrieved_docs = sparse_retriever.invoke(test_query)
for doc in retrieved_docs:
    print(f"내용: {doc.page_content}")
    print('-'*80)

내용: 질문: 스마트 허브 모델 SH-M200X의 전원 오류 코드 ERR-502가 표시되면 어떻게 조치해야 하나요?
답변: ERR-502는 어댑터 전압 불균형 오류입니다. 제품 후면의 리셋(Reset) 버튼을 10초간 누른 후, 제공된 전용 12V/2A 정품 어댑터로 교체하여 재부팅해 주세요. 문제가 지속될 경우 A/S 센터(1588-0000)로 접수해 주시기 바랍니다.
--------------------------------------------------------------------------------
내용: 질문: 스마트 카메라 CAM-PRO-300의 SD 카드 인식 불가 오류 SD-301 조치법은 무엇인가요?
답변: SD-301은 파일 시스템 호환성 오류입니다. PC에서 SD 카드를 FAT32 포맷으로 재설정한 후 기기에 재장착하거나, 앱 내 [장치 관리] > [저장소 포맷] 기능을 이용해 초기화해 주세요.
--------------------------------------------------------------------------------


In [7]:
print('정리 노트 실행 :', [d.metadata['qa'] for d in sparse_retriever.invoke('ERR-502 오류코드 조치법')], '← 수업과 같은 1번 · 6번 (BM25 는 임베딩이 없어 결과가 같다)')

정리 노트 실행 : [1, 6] ← 수업과 같은 1번 · 6번 (BM25 는 임베딩이 없어 결과가 같다)


In [8]:
# 🧪 OpenAI 임베딩 대신 로컬 한국어 임베딩(jhgan/ko-sroberta-multitask)으로 실제 실행 — 순위 · 점수는 text-embedding-3-small 과 다를 수 있다

from langchain_huggingface import HuggingFaceEmbeddings
local_emb = HuggingFaceEmbeddings(model_name='jhgan/ko-sroberta-multitask',      # 처음 한 번 약 440MB 내려받음
                                  encode_kwargs={'normalize_embeddings': True})  # 길이 1 → 내적 = 코사인
print('벡터 길이 :', len(local_emb.embed_query('테스트')))

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

벡터 길이 : 768


In [9]:
# 🧪 OpenAI 임베딩 대신 로컬 한국어 임베딩(jhgan/ko-sroberta-multitask)으로 실제 실행 — 순위 · 점수는 text-embedding-3-small 과 다를 수 있다

vectorstore = Chroma.from_documents(documents=docs, embedding=local_emb, persist_directory='chroma_db_tech_support_local',
                                    collection_configuration={'hnsw': {'space': 'cosine'}})
dense_retriever = vectorstore.as_retriever(search_kwargs={'k': 2})
hybrid_retriever = EnsembleRetriever(retrievers=[sparse_retriever, dense_retriever], weights=[0.5, 0.5])

QUESTIONS = [('ERR-502 오류코드 조치법', 1), ('SD-301 해결법', 6), ('NET-104 떠요', 4), ('제14조 2항', 3),
             ('기계가 아무 반응이 없어요 다시 켜는 법', 5), ('와이파이가 자꾸 끊겨요', 2), ('산 지 열흘 됐는데 고장나서 돈 돌려받고 싶어요', 8),
             ('카드 결제 바꾸고 싶어', 7), ('메모리 카드가 안 읽혀요', 6), ('현관 비밀번호 잊어버렸어요', 3)]
nums = lambda ds: [d.metadata['qa'] for d in ds]
table = pd.DataFrame([{'질문': q, '정답 QA': a, '희소(BM25)': nums(sparse_retriever.invoke(q)),
                       '밀집(임베딩)': nums(dense_retriever.invoke(q)), '하이브리드': nums(hybrid_retriever.invoke(q))} for q, a in QUESTIONS])
table

,질문,정답 QA,희소(BM25),밀집(임베딩),하이브리드
0,ERR-502 오류코드 조치법,1,"[1, 6]","[1, 6]","[1, 6]"
1,SD-301 해결법,6,"[6, 1]","[6, 1]","[6, 1]"
2,NET-104 떠요,4,"[4, 6]","[4, 2]","[4, 6, 2]"
3,제14조 2항,3,"[3, 8]","[3, 8]","[3, 8]"
4,기계가 아무 반응이 없어요 다시 켜는 법,5,"[5, 6]","[5, 1]","[5, 6, 1]"
5,와이파이가 자꾸 끊겨요,2,"[2, 8]","[2, 4]","[2, 8, 4]"
6,산 지 열흘 됐는데 고장나서 돈 돌려받고 싶어요,8,"[8, 7]","[8, 1]","[8, 7, 1]"
7,카드 결제 바꾸고 싶어,7,"[7, 6]","[7, 6]","[7, 6]"
8,메모리 카드가 안 읽혀요,6,"[6, 3]","[6, 4]","[6, 3, 4]"
9,현관 비밀번호 잊어버렸어요,3,"[3, 8]","[3, 4]","[3, 8, 4]"


- 문서가 8개뿐이고 질문마다 단어가 하나쯤은 겹쳐서, 이 매뉴얼에서는 **세 방식 모두** 1등이 정답이다 — 차이를 보려면 §4 처럼 **비슷한 코드가 여러 개** 있거나 **겹치는 단어가 없는** 경우가 필요하다
- 하이브리드 결과가 2 ~ 3개인 것은 두 검색기가 2개씩 찾은 것을 **합치고 중복을 뺀** 결과라서다(수업 출력 '총 3개 문서 수집' 과 같다)

## 3. RRF 손계산 = `EnsembleRetriever` (교재 p.64)

$$\text{RRF}(d) = \sum_{i} \frac{w_i}{c + \text{rank}_i(d)} \qquad (c = 60,\ \text{순위는 1부터})$$

In [10]:
def rrf(rank_lists, weights, c=60):
    scores = {}
    for ranks, w in zip(rank_lists, weights):
        for r, doc_id in enumerate(ranks, start=1):
            scores[doc_id] = scores.get(doc_id, 0) + w / (c + r)
    return sorted(scores.items(), key=lambda kv: -kv[1])

print('EnsembleRetriever 의 c :', hybrid_retriever.c)
for q in ['산 지 열흘 됐는데 고장나서 돈 돌려받고 싶어요', '와이파이가 자꾸 끊겨요']:
    s, d = nums(sparse_retriever.invoke(q)), nums(dense_retriever.invoke(q))
    mine = rrf([s, d], [0.5, 0.5])
    print(f'\n{q}\n  희소 {s} · 밀집 {d}')
    print('  손계산 :', [(n, round(v, 5)) for n, v in mine])
    print('  Ensemble :', nums(hybrid_retriever.invoke(q)), '| 같다 :', [n for n, _ in mine] == nums(hybrid_retriever.invoke(q)))

EnsembleRetriever 의 c : 60

산 지 열흘 됐는데 고장나서 돈 돌려받고 싶어요
  희소 [8, 7] · 밀집 [8, 1]
  손계산 : [(8, 0.01639), (7, 0.00806), (1, 0.00806)]
  Ensemble : [8, 7, 1] | 같다 : True



와이파이가 자꾸 끊겨요
  희소 [2, 8] · 밀집 [2, 4]
  손계산 : [(2, 0.01639), (8, 0.00806), (4, 0.00806)]
  Ensemble : [2, 8, 4] | 같다 : True


In [11]:
# 교재 p.64 의 순서 예시 : (키워드 1등 + 벡터 2등) > (키워드 3등 + 벡터 1등) > (키워드 2등 + 벡터 3등)
example = {'A (1등 + 2등)': (1, 2), 'B (3등 + 1등)': (3, 1), 'C (2등 + 3등)': (2, 3)}
for name, (r1, r2) in example.items():
    print(f'{name} : 1/(60+{r1}) + 1/(60+{r2}) = {1/(60+r1) + 1/(60+r2):.5f}')

A (1등 + 2등) : 1/(60+1) + 1/(60+2) = 0.03252
B (3등 + 1등) : 1/(60+3) + 1/(60+1) = 0.03227
C (2등 + 3등) : 1/(60+2) + 1/(60+3) = 0.03200


- 교재 예시의 순서는 계산해도 그대로다. 차이가 아주 작다(0.0325 · 0.0323 · 0.0320) — $c = 60$ 이 등수 차이를 부드럽게 만든다
- BM25 점수(예 : 2.06)와 코사인(예 : 0.65)은 **척도가 달라서** 그냥 더할 수 없다 → RRF 는 점수를 버리고 **등수**만 쓴다 (교재 p.64 ① ~ ③). 교재의 '밀집 검색은 0 ~ 1 사이' 는 대개 그렇지만 코사인은 −1 까지 가능하고, Chroma 기본값은 **거리**(작을수록 가까움)다
- 같은 문서가 한쪽에만 있으면 그쪽 점수만 받는다 — **양쪽 모두에서 상위**인 문서가 1등

## 4. 단일 검색의 문제 — 비슷한 코드 · 겹치는 단어 없음 (교재 p.63)

교재 p.63 의 예(X-2201 vs X-2202 · '컴퓨터 화면이 안 켜져요' ↔ '모니터 전원 공급 불량')를 확인하려고 **연습용 문서 6개**를 만들었다(실제 매뉴얼이 아니다).

In [12]:
# 🧪 OpenAI 임베딩 대신 로컬 한국어 임베딩(jhgan/ko-sroberta-multitask)으로 실제 실행 — 순위 · 점수는 text-embedding-3-small 과 다를 수 있다

error_texts = ['오류 코드 X-2201 : 냉각 팬 회전 불량입니다. 팬 커넥터를 다시 꽂고 먼지를 제거하세요.',
               '오류 코드 X-2202 : 온도 센서 단선입니다. 센서 케이블을 새것으로 교체하세요.',
               '오류 코드 X-2210 : 전원부 과전류입니다. 전원 케이블을 뽑고 5분 뒤 다시 연결하세요.',
               '오류 코드 X-2301 : 펌웨어 서명 검증 실패입니다. 최신 펌웨어로 업데이트하세요.',
               '모니터 전원 공급 불량 : 모니터 뒤쪽 전원 어댑터와 케이블 연결을 확인하세요.',
               '프린터 용지 걸림 : 뒷면 덮개를 열고 걸린 용지를 천천히 당겨 빼세요.']
error_docs = [Document(t, metadata={'no': i + 1}) for i, t in enumerate(error_texts)]
err_sparse = BM25Retriever.from_documents(error_docs, preprocess_func=mecab_tokenizer, k=3)
err_dense = Chroma.from_documents(error_docs, local_emb, persist_directory='chroma_db_error_codes_local',
                                  collection_configuration={'hnsw': {'space': 'cosine'}}).as_retriever(search_kwargs={'k': 3})
err_hybrid = EnsembleRetriever(retrievers=[err_sparse, err_dense], weights=[0.5, 0.5])
no = lambda ds: [d.metadata['no'] for d in ds]
rows = []
for q, ans in [('X-2201 원인이 뭐야?', 1), ('X-2202 조치 방법', 2), ('X-2210', 3), ('컴퓨터 화면이 안 켜져요', 5), ('종이가 끼었어요', 6)]:
    bm = err_sparse.vectorizer.get_scores(mecab_tokenizer(q))
    rows.append({'질문': q, '정답': ans, '희소': no(err_sparse.invoke(q)), 'BM25 점수 최대': round(bm.max(), 2),
                 '밀집': no(err_dense.invoke(q)), '하이브리드': no(err_hybrid.invoke(q))})
pd.DataFrame(rows)

,질문,정답,희소,BM25 점수 최대,밀집,하이브리드
0,X-2201 원인이 뭐야?,1,"[1, 4, 2]",1.64,"[1, 2, 3]","[1, 2, 4, 3]"
1,X-2202 조치 방법,2,"[2, 4, 3]",1.73,"[1, 2, 3]","[2, 3, 1, 4]"
2,X-2210,3,"[3, 4, 2]",1.64,"[2, 1, 3]","[3, 2, 4, 1]"
3,컴퓨터 화면이 안 켜져요,5,"[6, 5, 4]",0.00,"[5, 6, 3]","[6, 5, 4, 3]"
4,종이가 끼었어요,6,"[6, 5, 4]",0.00,"[6, 5, 1]","[6, 5, 4, 1]"


- **밀집만** : 'X-2202 조치 방법' → 1등이 **X-2201** 문서, 'X-2210' → 1등이 **X-2202** — 문장 모양이 똑같고 숫자만 다른 문서를 임베딩은 거의 구별하지 못한다(교재 p.63 ①)
- **희소만** : '컴퓨터 화면이 안 켜져요' · '종이가 끼었어요' → 겹치는 단어가 없어 BM25 점수가 **모두 0** — 그래도 3개(뒤에서부터 6 · 5 · 4번)를 돌려준다(교재 p.63 ②)
- **하이브리드** : 코드 질문은 희소 덕분에 정답이 1등으로 올라온다. 그런데 '컴퓨터 화면이 안 켜져요' 는 밀집이 맞힌 5번과 **0점 희소의 1등**(6번)이 같은 RRF 점수가 되어 순서가 앞선 6번이 1등이 됐다 — 점수 0 인 결과도 '1등' 으로 계산되기 때문

## 5. 고쳐 보기 — ① 0점 희소 결과 빼기 · ② 코드를 통째로 토큰으로

In [13]:
# 🧪 OpenAI 임베딩 대신 로컬 한국어 임베딩(jhgan/ko-sroberta-multitask)으로 실제 실행 — 순위 · 점수는 text-embedding-3-small 과 다를 수 있다

import re
from langchain_core.runnables import RunnableLambda

def positive_only(retriever):
    '''BM25 점수가 0 보다 큰 문서만 (같은 점수는 원래 순서대로)'''
    def search(query):
        scores = retriever.vectorizer.get_scores(retriever.preprocess_func(query))
        return [retriever.docs[i] for i in np.argsort(-scores, kind='stable') if scores[i] > 0][:retriever.k]
    return RunnableLambda(search)

CODE = re.compile(r'[A-Za-z]+[A-Za-z0-9]*(?:-[A-Za-z0-9]+)+')
KEEP = ('NN', 'VV', 'VA', 'XR', 'SL', 'SN', 'MAG')
def code_aware_tokenizer(text):                       # 코드는 통째로 · 문장부호는 버림 (bm25_sparse_search §5)
    codes = [c.upper() for c in CODE.findall(text)]
    return codes + [m.lower() for m, pos in mecab.pos(CODE.sub(' ', text)) if pos.startswith(KEEP)]

def three_ways(docs, dense, k):
    base = BM25Retriever.from_documents(docs, preprocess_func=mecab_tokenizer, k=k)
    code = BM25Retriever.from_documents(docs, preprocess_func=code_aware_tokenizer, k=k)
    return {'수업 방식': EnsembleRetriever(retrievers=[base, dense], weights=[0.5, 0.5]),
            '① 0점 제외': EnsembleRetriever(retrievers=[positive_only(base), dense], weights=[0.5, 0.5]),
            '② 코드 토큰 + 0점 제외': EnsembleRetriever(retrievers=[positive_only(code), dense], weights=[0.5, 0.5])}

ERROR_QS = [('X-2201 원인이 뭐야?', 1), ('X-2202 조치 방법', 2), ('X-2210', 3), ('컴퓨터 화면이 안 켜져요', 5), ('종이가 끼었어요', 6)]
key = lambda d: d.metadata.get('no', d.metadata.get('qa'))
rows = []
for set_name, ways, qs in [('연습용 오류 코드 5문항', three_ways(error_docs, err_dense, 3), ERROR_QS),
                           ('기술 지원 매뉴얼 10문항', three_ways(docs, dense_retriever, 2), QUESTIONS)]:
    for q, ans in qs:
        row = {'문항': set_name, '질문': q, '정답': ans}
        for name, r in ways.items():
            top = [key(d) for d in r.invoke(q)]
            row[name] = f"{'○' if top[0] == ans else '×'} {top}"
        rows.append(row)
result = pd.DataFrame(rows)
summary = result.groupby('문항')[['수업 방식', '① 0점 제외', '② 코드 토큰 + 0점 제외']].agg(lambda c: f"{sum(v.startswith('○') for v in c)}/{len(c)}")
print(summary.to_string(), '\n(1등이 정답인 질문 수)')
result

                수업 방식 ① 0점 제외 ② 코드 토큰 + 0점 제외
문항                                           
기술 지원 매뉴얼 10문항  10/10   10/10            9/10
연습용 오류 코드 5문항     4/5     4/5             5/5 
(1등이 정답인 질문 수)


,문항,질문,정답,수업 방식,① 0점 제외,② 코드 토큰 + 0점 제외
0,연습용 오류 코드 5문항,X-2201 원인이 뭐야?,1,"○ [1, 2, 4, 3]","○ [1, 2, 4, 3]","○ [1, 2, 3]"
1,연습용 오류 코드 5문항,X-2202 조치 방법,2,"○ [2, 3, 1, 4]","○ [2, 1, 4, 3]","○ [2, 1, 3]"
2,연습용 오류 코드 5문항,X-2210,3,"○ [3, 2, 4, 1]","× [2, 3, 1, 4]","○ [3, 2, 1]"
3,연습용 오류 코드 5문항,컴퓨터 화면이 안 켜져요,5,"× [6, 5, 4, 3]","○ [5, 6, 3]","○ [5, 6, 3]"
4,연습용 오류 코드 5문항,종이가 끼었어요,6,"○ [6, 5, 4, 1]","○ [6, 5, 1]","○ [6, 5, 1]"
5,기술 지원 매뉴얼 10문항,ERR-502 오류코드 조치법,1,"○ [1, 6]","○ [1, 6]","○ [1, 6]"
6,기술 지원 매뉴얼 10문항,SD-301 해결법,6,"○ [6, 1]","○ [6, 1]","○ [6, 1]"
7,기술 지원 매뉴얼 10문항,NET-104 떠요,4,"○ [4, 6, 2]","○ [4, 6, 2]","○ [4, 2]"
8,기술 지원 매뉴얼 10문항,제14조 2항,3,"○ [3, 8]","○ [3, 8]","○ [3, 8]"
9,기술 지원 매뉴얼 10문항,기계가 아무 반응이 없어요 다시 켜는 법,5,"○ [5, 6, 1]","○ [5, 6, 1]","○ [5, 6, 1]"


- 15문항 중 1등이 정답인 수는 **세 방식 모두 14개** — 고친 것이 공짜가 아니다
- ① **0점 제외** : '컴퓨터 화면이 안 켜져요' 는 고쳐졌다(밀집 결과만 남음). 대신 'X-2210' 이 틀렸다 — 수업 토큰화에서는 `X` · `-` 가 모든 X 코드 문서에 있어 X-2202(2번) · X-2301(4번) 문서가 BM25 **동점**(0.45)이다. 수업 방식은 4번을, 내 함수는 2번을 희소 2등에 놓았고, 밀집 1등인 2번이 희소 2등까지 받아 RRF 1등이 됐다 — 동점의 순서 하나로 답이 바뀐다
- ② **코드 토큰 + 0점 제외** : 연습용 5문항은 모두 맞는다 — `X-2210` 은 그 문서에만 있는 토큰이라 희소 결과가 1개뿐. 대신 매뉴얼의 '메모리 카드가 안 읽혀요' 가 틀렸다 — 남기기로 한 부사(`MAG`) **`안`** 이 NET-104 문서의 '연결 **안** 됨' 과 겹쳤다
- 그래서 **질문 묶음을 정해 두고, 고칠 때마다 같은 묶음으로 다시 잰다.** 5 · 10문항은 너무 적어 어느 쪽이 낫다고 말할 수 없다 — 실제 서비스라면 질문 수십 ~ 수백 개로 잰다
- `EnsembleRetriever` 는 검색기 대신 `RunnableLambda`(질문 → Document 리스트)도 받는다. 가중치(`weights`)를 바꾸는 방법도 있지만, 0점 결과가 '1등' 으로 계산되는 문제는 가중치로는 사라지지 않는다

## 6. RAG 체인 · 챗봇

In [14]:
system_prompt = '''
당신은 스마트홈 IT 기기 및 지원 서비스의 전문 기술 지원 AI 엔지니어입니다.
제공된 [참고 컨텍스트] 정보를 바탕으로 고객의 제품 문의, 오류 코드 해결법, A/S 규정 질문에 정중하고 명확하게 답변해주세요.

[답변 작성 수칙]
1. [참고 컨텍스트]에 명시된 모델명, 오류 코드, 조치 단계, 규정 조항을 정확하게 인용하세요.
2. 답변을 시작할 때 '답변:'이라는 단어를 사용하지 마세요.
3. [참고 컨텍스트]에 고객 질문에 관한 내용이 없다면, "죄송합니다. 문의하신 내용에 대한 기술 매뉴얼 정보를 찾을 수 없습니다. 고객센터(1588-0000)로 문의 부탁드립니다."라고 답변하세요.

[참고 컨텍스트]
{context}
'''
prompt = ChatPromptTemplate.from_messages([('system', system_prompt), ('human', '{question}')])

def ask(question, chain, retriever):
    docs = retriever.invoke(question)
    context = '\n\n'.join(d.page_content for d in docs)
    return chain.invoke({'context': context, 'question': question}), [d.metadata['qa'] for d in docs]

In [ ]:
# 📼 아래 출력 = 수업 시간에 실행한 결과 (필기 원본에서 옮김)

# 챗봇 while 반복문 (질문 3번 · input() 으로 넣은 질문 글자는 기록에 남지 않음)

안녕하세요! 하이브리드 RAG 기반 고객 응대 챗봇입니다. 궁금한 점을 질문해주세요. (종료하려면 'exit' 입력)
--------------------------------------------------------------------------------
하이브리드 검색 결과: 총 3개 문서 수집
--------------------------------------------------------------------------------
답변의 근거: 
질문: 스마트 허브 모델 SH-M200X의 전원 오류 코드 ERR-502가 표시되면 어떻게 조치해야 하나요?
답변: ERR-502는 어댑터 전압 불균형 오류입니다. 제품 후면의 리셋(Reset) 버튼을 10초간 누른 후, 제공된 전용 12V/2A 정품 어댑터로 교체하여 재부팅해 주세요. 문제가 지속될 경우 A/S 센터(1588-0000)로 접수해 주시기 바랍니다.

질문: 스마트 카메라 CAM-PRO-300의 SD 카드 인식 불가 오류 SD-301 조치법은 무엇인가요?
답변: SD-301은 파일 시스템 호환성 오류입니다. PC에서 SD 카드를 FAT32 포맷으로 재설정한 후 기기에 재장착하거나, 앱 내 [장치 관리] > [저장소 포맷] 기능을 이용해 초기화해 주세요.

질문: 앱에서 네트워크 연결 안 됨 상태로 표시되고 코드 NET-104가 뜹니다.
답변: NET-104 오류는 방화벽 또는 포트 차단 문제입니다. 사용 중인 공유기의 8080 및 443 포트 포워딩 상태를 확인하시고, 공유기 재부팅 후 앱을 다시 실행해 주세요.
--------------------------------------------------------------------------------
챗봇 응답: 
스마트 허브 SH-M200X의 ERR-502 전원 오류에 대한 조치법은 다음과 같습니다.

- ERR-502는 어댑터 전압 불균형 오류입니다.
- 1) 제품 후면의 리셋(Reset) 버튼을 10초간 누릅니다.
- 2) 제공

In [16]:
# 🧪 LLM 대신 '받은 컨텍스트의 첫 줄을 되돌려 주는' 가짜 LLM — 체인에 무엇이 들어가는지 확인용

from langchain_core.runnables import RunnableLambda
from langchain_core.messages import AIMessage

def _fake_llm(prompt_value):
    system = prompt_value.to_messages()[0].content
    context = system.split('[참고 컨텍스트]')[-1].strip()
    first = context.splitlines()[0] if context else '(컨텍스트 없음)'
    return AIMessage(f'(모의 답변 · 컨텍스트 {len(context)}자를 받음) {first[:70]}')

fake_llm = RunnableLambda(_fake_llm)

In [17]:
# 🧪 OpenAI 임베딩 대신 로컬 한국어 임베딩(jhgan/ko-sroberta-multitask)으로 실제 실행 — 순위 · 점수는 text-embedding-3-small 과 다를 수 있다
#   + LLM 대신 '받은 컨텍스트의 첫 줄을 되돌려 주는' 가짜 LLM — 체인에 무엇이 들어가는지 확인용

fake_chain = prompt | fake_llm | StrOutputParser()
for q in ['ERR-502 오류코드 조치법', '기기가 먹통이에요 강제로 재부팅하려면?', 'DL-900 비상 비밀번호 초기화']:
    answer, used = ask(q, fake_chain, hybrid_retriever)
    print(f'질문 : {q}\n  근거 QA : {used}\n  답 : {answer}')

질문 : ERR-502 오류코드 조치법
  근거 QA : [1, 6]
  답 : (모의 답변 · 컨텍스트 377자를 받음) 질문: 스마트 허브 모델 SH-M200X의 전원 오류 코드 ERR-502가 표시되면 어떻게 조치해야 하나요?
질문 : 기기가 먹통이에요 강제로 재부팅하려면?
  근거 QA : [5, 6, 4]
  답 : (모의 답변 · 컨텍스트 466자를 받음) 질문: 기기가 완전히 먹통이 되고 화면이 아무 반응도 없을 때 강제 재부팅하는 방법이 있나요?


질문 : DL-900 비상 비밀번호 초기화
  근거 QA : [3, 6]
  답 : (모의 답변 · 컨텍스트 357자를 받음) 질문: 도어락 모델 DL-900 시리즈의 비상 비밀번호 초기화 규정(제14조 2항)은 어떻게 되나요?


In [ ]:
# ⏳ API 키가 필요한 셀 — 정리 환경에서는 실행하지 않음 (직접 실행해 보기)

openai_dense = Chroma.from_documents(docs, OpenAIEmbeddings(model='text-embedding-3-small'),
                                     persist_directory='chroma_db_tech_support_openai').as_retriever(search_kwargs={'k': 2})
openai_hybrid = EnsembleRetriever(retrievers=[sparse_retriever, openai_dense], weights=[0.5, 0.5])
rag_chain = prompt | ChatOpenAI(model='gpt-5-nano') | StrOutputParser()
for q in ['ERR-502 오류코드 조치법', 'ERR-503 오류는 뭔가요?', '와이파이가 자꾸 끊겨요', '프린터 용지가 걸렸어요']:
    answer, used = ask(q, rag_chain, openai_hybrid)
    print(f'질문 : {q}\n  근거 QA : {used}\n  답 : {answer}\n')

- 수업 기록 3번 모두 답이 근거 QA 와 맞다(ERR-502 조치 3단계 · 강제 하드 리셋 7초 · DL-900 제14조 2항). 하이브리드는 매번 **3개**를 모아 넘겼고, 그중 하나만 질문과 관련 있었다 — 나머지 2개는 LLM 이 무시해야 하는 글이다
- ⏳ 셀의 'ERR-503' · '프린터' 는 매뉴얼에 없는 질문이다 — 수칙 3 대로 '찾을 수 없습니다' 가 나오는지 확인해 보기

> **📝 고친 부분 (필기 `RAG_실습5_1` · 교재)**
> - 토큰화 함수의 `S` 는 문장부호(`SF` · `SY` · `SC`)까지 토큰으로 남긴다 · `ERR-502` 를 `ERR` · `-` · `502` 로 조각냄 → 매뉴얼에 없는 `ERR-503` 질문에도 ERR-502 문서가 1등([`bm25_sparse_search.ipynb`](./bm25_sparse_search.ipynb) §5)
> - BM25 점수가 모두 0 이어도 k개를 돌려줘 하이브리드 순위를 흐린다 → 0점 제외 · 코드 토큰을 시험(§5) — 각각 고치는 질문과 망가뜨리는 질문이 있었다
> - `chunk_size=210` 은 최댓값 208 + 구분자 2 에 딱 맞춘 값 — QA 가 늘면 다시 잰다
> - '폴더가 있으면 불러오기'(`chroma_db_tech_support`) · `while True … except Exception: print` 로 오류가 나도 계속 도는 구조 → Day 049 와 같이 고침
> - 교재 p.64 '척도가 **달러서**' · '**백터** 1등' → 달라서 · 벡터 / '밀집 검색은 0 ~ 1 사이의 실수' → 코사인은 −1 ~ 1, Chroma 기본은 거리(작을수록 가까움)
> - 교재 p.62 표의 밀집 검색 약점 '**환각**' → 환각은 LLM 이 만든다. 밀집 검색의 약점은 **비슷하지만 틀린 문서**(X-2201 ↔ X-2202)를 가져오는 것이고, 그것이 LLM 의 틀린 답(환각)으로 이어진다

## 정리

| | 희소 (BM25) | 밀집 (임베딩) | 하이브리드 (RRF) |
|---|---|---|---|
| 잘 찾는 것 | 코드 · 조항 · 고유명사 (`X-2202`) | 다른 말로 한 같은 뜻 ('화면이 안 켜져요') | 둘 다 |
| 못 찾는 것 | 겹치는 단어가 없는 질문 — 0점이어도 k개 반환 | 숫자만 다른 비슷한 문서 | 0점 희소 결과 · 동점이 섞이면 흐려짐 → 고친 뒤 같은 질문 묶음으로 다시 잰다 |
| 코드 | `BM25Retriever.from_documents(docs, preprocess_func=…, k=2)` | `vectorstore.as_retriever(k=2)` | `EnsembleRetriever(retrievers=[…], weights=[0.5, 0.5])` · $c = 60$ |